# LoRA Fine-tuning (Google Colab 版)

在 **Google Colab** 上微調 **SmolLM2-1.7B-Instruct**，用於 NBA 球員數據分析任務。

## 使用前：Google Drive 設定

### 1. 建立資料夾結構

在 Google Drive 根目錄建立一個專案資料夾（例如 `genai_hw3`），結構如下：

```
My Drive/
└── genai_hw3/           ← 專案根目錄（可自訂名稱）
    ├── data/            ← 放訓練資料
    │   └── train.jsonl   ← 必須！把 train.jsonl 上傳到這裡
    └── outputs/         ← 訓練完成後 adapter 會存到這裡（會自動建立）
        └── smollm2-bet-advisor/
```

### 2. 上傳 train.jsonl

- 路徑：`My Drive/genai_hw3/data/train.jsonl`
- 若資料夾不存在，先在 Drive 建立 `genai_hw3` 和 `data`，再把 `train.jsonl` 上傳到 `data` 裡

### 3. 修改路徑（若你用了不同資料夾名稱）

在下方「Path Configuration」的 cell 中，把 `PROJECT_FOLDER = "genai_hw3"` 改成你的資料夾名稱。

## 0. 安裝依賴 & 啟用 GPU

In [ ]:
# Colab 選單：Runtime → Change runtime type → Hardware accelerator: GPU (T4)
!pip install -q transformers peft datasets accelerate

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (use CPU)'}")

## 1. 掛載 Google Drive & Path Configuration

In [ ]:
from google.colab import drive
from pathlib import Path

# 掛載 Google Drive（會跳出授權視窗，點選你的帳號並允許）
drive.mount("/content/drive")

# ========== 請依你的 Drive 結構修改 ==========
# 專案資料夾名稱（在 My Drive 根目錄下）
PROJECT_FOLDER = "genai_hw3"

# 專案根目錄：/content/drive/MyDrive/<PROJECT_FOLDER>
project_root = Path("/content/drive/MyDrive") / PROJECT_FOLDER
data_dir = project_root / "data"
output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / "train.jsonl"
adapter_path = output_dir / "smollm2-bet-advisor"

print(f"Project root: {project_root}")
print(f"Train: {train_path} (exists: {train_path.exists()})")
print(f"Adapter output: {adapter_path}")

if not train_path.exists():
    raise FileNotFoundError(
        f"找不到 train.jsonl！請確認：\n"
        f"  1. 已掛載 Google Drive\n"
        f"  2. 路徑正確：{train_path}\n"
        f"  3. 已上傳 train.jsonl 到 data/ 資料夾"
    )

## 2. Load Base Model and Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Loading model: {MODEL_ID}")
# Colab T4 支援 bf16，可省記憶體
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

if not torch.cuda.is_available():
    model = model.to("cpu")

print(f"Model loaded. Device: {next(model.parameters()).device}")

## 3. System Prompt & Load Training Data

In [ ]:
import json

SYSTEM_PROMPT = """You are an expert NBA player data analyst. Your task is to analyze betting questions (over/under) using historical player statistics.

## Tree of Thought Reasoning Framework

Build your reasoning as a tree with these main branches (evaluate each, then synthesize):

1. **Sample & Statistics Branch**: For each context filter (all games, last N games, starter vs bench, with/without star teammates), assess:
   - n_games: Is sample size sufficient? (n < 10 → low weight)
   - p_over, p_under, mean, std: Which context favors over vs under?
   - Conflict between contexts → note uncertainty

2. **Lineup/Teammate Branch**: How does starter vs bench, or with/without star teammates, change the stats? Does lineup context support or contradict the main trend?

3. **Risk & Synthesis Branch**: Given the above, what is the net signal? If branches conflict or sample is weak → prefer "avoid".

## Output Format (JSON only, no markdown)

{
  "decision": "over" | "under" | "avoid",
  "confidence": 0.0 to 1.0,
  "reasoning": {
    "tree_of_thought": [
      {"step": 1, "branch": "sample_stats", "thought": "...", "conclusion": "..."},
      {"step": 2, "branch": "lineup_teammate", "thought": "...", "conclusion": "..."},
      {"step": 3, "branch": "synthesis", "thought": "...", "conclusion": "..."}
    ]
  },
  "summary": "One-sentence conclusion"
}

Each step must have: branch (which dimension), thought (your analysis), conclusion (what this branch implies for the decision).
Respond with ONLY valid JSON, no markdown or extra text."""

def load_jsonl(path: Path) -> list[dict]:
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

train_data = load_jsonl(train_path)
print(f"Train: {len(train_data)} records")
if train_data:
    ex = train_data[0]
    print(f"  instruction: {ex['instruction'][:60]}...")
    print(f"  input: {len(ex['input'])} chars, output: {len(ex['output'])} chars")

## 4. Prepare Dataset for Training

In [ ]:
MAX_SEQ_LENGTH = 4096

def format_training_example(item: dict) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{item['instruction']}\n\nStatistics:\n{item['input']}"},
        {"role": "assistant", "content": item["output"]},
    ]
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    prompt_messages = messages[:-1]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)
    prompt_len = prompt_ids.input_ids.shape[1]

    full_ids = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    input_ids = full_ids.input_ids[0]
    labels = input_ids.clone()
    labels[:prompt_len] = -100

    return {"input_ids": input_ids, "labels": labels, "attention_mask": full_ids.attention_mask[0]}

tokenized = [format_training_example(item) for item in train_data]
print(f"Tokenized {len(tokenized)} examples")
print(f"Example input_ids length: {len(tokenized[0]['input_ids'])}")

## 5. Create Dataset & Data Collator

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

def to_dataset_format(tok_list):
    return {
        "input_ids": [t["input_ids"].tolist() for t in tok_list],
        "labels": [t["labels"].tolist() for t in tok_list],
        "attention_mask": [t["attention_mask"].tolist() for t in tok_list],
    }

ds_dict = to_dataset_format(tokenized)
train_dataset = Dataset.from_dict(ds_dict)

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt",
    label_pad_token_id=-100,
)

print(f"Dataset: {train_dataset}")

## 6. LoRA Config & PEFT Model

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 省記憶體（Colab T4 建議開啟）
model.gradient_checkpointing_enable()
model.config.use_cache = False

## 7. Training Arguments & Trainer

In [ ]:
from transformers import TrainingArguments, Trainer

# Colab T4 (16GB) 建議 batch_size=2，若 OOM 可改為 1
training_args = TrainingArguments(
    output_dir=str(adapter_path),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

## 8. Save Adapter to Google Drive

In [ ]:
trainer.save_model(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))
print(f"Saved adapter and tokenizer to: {adapter_path}")
print("\n可在 Google Drive 的 outputs/smollm2-bet-advisor/ 找到：adapter_config.json, adapter_model.safetensors, tokenizer 等檔案")